# Experimental Evaluation of Hyperparameter Variations in an Existing SVM-Based sEMG QWERTY Keypress Classification Model

This notebook reuses the ANSLab-UHN feature pipeline and RBF-SVM baseline. It evaluates `C` and `gamma` reproducibly on the included P1 sample.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'svm_experiment':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project: {PROJECT_ROOT}')
print(f'Python:  {sys.executable}')

## One-time data preparation
Run the next cell once after cloning. It converts the included P1 `.rhd` recordings into the original 96-feature vectors.

In [ ]:
from svm_experiment.prepare_sample_data import prepare_pipeline

feature_dir = PROJECT_ROOT / 'CleanData' / 'valid_user_features'
if not (feature_dir / 'P1_T1_X.npy').exists():
    prepare_pipeline(data_root=PROJECT_ROOT / 'CleanData')
else:
    print(f'Prepared data already exists: {feature_dir}')

## Run the SVM hyperparameter experiment
Edit the value lists below to explore different RBF-SVM configurations. Outputs and fitted models are saved under `outputs/`.

In [ ]:
from svm_experiment.run_hyperparameter_experiment import run_experiments

results = run_experiments(
    data_dir=PROJECT_ROOT / 'CleanData' / 'valid_user_features',
    output_dir=PROJECT_ROOT / 'outputs',
    participant='P1',
    mode='both',
    C_values=(0.1, 1.0, 10.0, 100.0),
    gamma_values=('scale', 'auto', 0.001, 0.01, 0.1),
    cv_folds=3,
    random_state=42,
    n_jobs=1,  # reliable default for Windows/Jupyter
)
results

In [ ]:
results.pivot(index='scenario', columns='variant', values=['accuracy', 'balanced_accuracy', 'f1_macro'])